# 00_setup_and_config

Configure Azure Databricks access to ADLS Gen2 using a service principal and direct `abfss://` paths. Replace the placeholder secret before running.

In [0]:
# ============================================================
# CONFIG
# ============================================================
# Fill in / verify these values before running the notebook.

PROJECT_NAME = "gaming-product-marketing-analytics"

# New ADLS Gen2 storage account
STORAGE_ACCOUNT_NAME = "gamingmarketd01"

# Container in the new storage account
CONTAINER_NAME = "gaming-data"

# Microsoft Entra / Service Principal values
CLIENT_ID = "client id"
TENANT_ID = "Tennant id"

# Paste the NEW client secret VALUE here
CLIENT_SECRET = "some-long-secret-value"

In [0]:
# ============================================================
# BUILD COMMON PATHS
# ============================================================
# These variables are intended to be reused by downstream notebooks.

STORAGE_FQDN = f"{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net"
BASE_PATH = f"abfss://{CONTAINER_NAME}@{STORAGE_FQDN}/"

RAW_PATH = f"{BASE_PATH}raw/"
BRONZE_PATH = f"{BASE_PATH}bronze/"
SILVER_PATH = f"{BASE_PATH}silver/"
GOLD_PATH = f"{BASE_PATH}gold/"

CHECKPOINT_PATH = f"{BASE_PATH}checkpoints/"
TEMP_PATH = f"{BASE_PATH}temp/"

print("Project:", PROJECT_NAME)
print("Storage account:", STORAGE_ACCOUNT_NAME)
print("Container:", CONTAINER_NAME)
print("Base path:", BASE_PATH)

In [0]:
# ============================================================
# SPARK ACCESS SETUP FOR ADLS GEN2
# ============================================================
# ADLS Gen2 stands for Azure Data Lake Storage Gen2.
# It is Microsoft Azure's cloud storage service for storing large amounts of data,
# such as CSV files, parquet files, images, logs, and analytics datasets.
#
# In this project, ADLS Gen2 is the place where your raw data files live.
# Spark needs permission before it can read files from that storage account.
#
# These settings tell Spark:
# - which authentication method to use,
# - which Azure service is providing the login,
# - and which storage account these settings apply to.
#
# We are using OAuth with a service principal.
# A service principal is like a machine account:
# it lets Databricks log in securely to Azure without using your personal username and password.
#
# STORAGE_FQDN means the full storage account address,
# for example: myaccount.dfs.core.windows.net
#
# In short:
# this config is what allows Spark in Databricks
# to securely connect to your ADLS Gen2 storage account.

spark.conf.set(
    f"fs.azure.account.auth.type.{STORAGE_FQDN}",
    "OAuth"
)

spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{STORAGE_FQDN}",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{STORAGE_FQDN}",
    CLIENT_ID
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{STORAGE_FQDN}",
    CLIENT_SECRET
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{STORAGE_FQDN}",
    f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/token"
)

print("Spark OAuth configuration applied successfully.")

In [0]:
# ============================================================
# CONNECTIVITY TEST
# ============================================================
# Test whether Databricks can access the ADLS Gen2 container root.

display(dbutils.fs.ls(BASE_PATH))

In [0]:
# ============================================================
# CREATE PROJECT FOLDERS
# ============================================================
# Safe to run multiple times.

dbutils.fs.mkdirs(RAW_PATH)
dbutils.fs.mkdirs(BRONZE_PATH)
dbutils.fs.mkdirs(SILVER_PATH)
dbutils.fs.mkdirs(GOLD_PATH)
dbutils.fs.mkdirs(CHECKPOINT_PATH)
dbutils.fs.mkdirs(TEMP_PATH)

print("Created / confirmed folder structure:")
print(" -", RAW_PATH)
print(" -", BRONZE_PATH)
print(" -", SILVER_PATH)
print(" -", GOLD_PATH)
print(" -", CHECKPOINT_PATH)
print(" -", TEMP_PATH)

In [0]:
# ============================================================
# VERIFY FOLDER STRUCTURE
# ============================================================

display(dbutils.fs.ls(BASE_PATH))

In [0]:
# ============================================================
# OPTIONAL HELPER VALUES FOR DOWNSTREAM NOTEBOOKS
# ============================================================
# If later notebooks use these variables, they can stay unchanged
# as long as they reference these names instead of hardcoded paths.

CATALOG_NAME = "main"
SCHEMA_NAME = "default"

print("Setup complete.")